# 3-qubit GST (`GST_model_3Q.ipynb`)

Same pipeline as that of single qubit model — **model -> design -> FPR -> shot sweep -> accuracy plots** — but built from a custom `QubitProcessorSpec` because pyGSTi ships **no 3-qubit model pack**.

## Still working on this notebook, 3 qubit GST might be more challenging.

In [1]:
# --- imports + local module paths (mirrors the 1-qubit notebook) ---
import sys, os, time, pickle, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

import pygsti
from pygsti.processors import QubitProcessorSpec
from pygsti.models import create_explicit_model
from pygsti.algorithms import fiducialselection as _fs, germselection as _gs
import pygsti.tools as _T
from pygsti.tools import entanglement_infidelity, diamonddist

_cwd = Path.cwd()
# gradient_pounders (optional drop-in solver) and the FPR module
for _root in (_cwd, _cwd.parent, _cwd.parent.parent):
    _pp = _root / "pounders" / "py"
    if (_pp / "gradient_pounders.py").exists() and str(_pp) not in sys.path:
        sys.path.insert(0, str(_pp))
for _d in (_cwd, _cwd / "GST_POUNDERS", _cwd.parent / "GST_POUNDERS"):
    if (_d / "near_minimal_fpr_reduction.py").exists() and str(_d) not in sys.path:
        sys.path.insert(0, str(_d))
import near_minimal_fpr_reduction as fprmod
try:
    import gradient_pounders as pounders   # optional; scipy is the default solver below
except Exception as _e:
    pounders = None
    print("gradient_pounders not imported (scipy least_squares will be used):", _e)
print("ready")

ready


In [2]:
# --- configuration ---
QUBITS       = 3
GATE_NAMES   = ["Gxpi2", "Gypi2", "Gcnot"]     # X(pi/2), Y(pi/2) on each qubit + CNOT along a line
GEOMETRY     = "line"                           # 0-1-2 connectivity
OP_NOISE     = 0.01                             # depolarizing on gates (truth)
SPAM_NOISE   = 0.005
MAX_LENGTHS  = [1, 2, 4]                         # start shallow; deepen once it runs
SHOT_SWEEP   = [1e3, 1e4, 1e5]
DATA_SEED    = 2024
OUTCOMES_PER_CIRCUIT = 2 ** QUBITS              # 8 computational-basis outcomes
CACHE_DIR    = Path("Tests3Q"); CACHE_DIR.mkdir(exist_ok=True)
# FPR stages (same as the 1-qubit study)
FPR_STAGE1   = "twirled_derivative"
FPR_STAGE2   = "paper_greedy"
print("config set:", QUBITS, "qubits,", GATE_NAMES, GEOMETRY)

config set: 3 qubits, ['Gxpi2', 'Gypi2', 'Gcnot'] line


In [3]:
# --- build the models  [VERIFIED]  ---
pspec = QubitProcessorSpec(QUBITS, GATE_NAMES, geometry=GEOMETRY)
ideal = create_explicit_model(pspec)                          # ideal target -> used for experiment design
truth = ideal.copy(); truth.set_all_parameterizations("full") # 'full' makes the gates settable (CPTPLND/H+S do not convert -- hurdle #2)
truth = truth.depolarize(op_noise=OP_NOISE, spam_noise=SPAM_NOISE)   # noisy data-generation model ("truth")

# FIT TEMPLATE.  NOTE (hurdle #2): CPTPLND/H+S do not convert for multi-qubit in this pyGSTi version.
# 'full TP' works but is ~41k params (impractical). For a runnable scaffold we fit a small local model
# (depolarization parameters) that CAN represent the depolarizing truth; upgrade to a crosstalk-free
# local-Lindblad model for coherent errors (see hurdle #2 in the intro).
from pygsti.models import create_crosstalk_free_model
fit_template = create_crosstalk_free_model(
    pspec,
    depolarization_strengths={g: 0.0 for g in GATE_NAMES},
    depolarization_parameterization="depolarize",
)
print("ideal ops        :", list(ideal.operations.keys()))
print("Hilbert-Schmidt dim:", ideal.dim, "(= 4^%d)" % QUBITS)
print("fit template params:", fit_template.num_params, " (depol-only demo; see hurdle #2 to enrich)")

ideal ops        : [Label(('Gxpi2', 0)), Label(('Gxpi2', 1)), Label(('Gxpi2', 2)), Label(('Gypi2', 0)), Label(('Gypi2', 1)), Label(('Gypi2', 2)), Label(('Gcnot', 0, 1)), Label(('Gcnot', 1, 0)), Label(('Gcnot', 1, 2)), Label(('Gcnot', 2, 1))]
Hilbert-Schmidt dim: 64 (= 4^3)
fit template params: 3  (depol-only demo; see hurdle #2 to enrich)


In [ ]:
# --- fiducials + germs (CACHED, EXPENSIVE)  [run on your compute; see hurdle #1] ---
_fid_pkl, _germ_pkl = CACHE_DIR / "fiducials.pkl", CACHE_DIR / "germs.pkl"
if _fid_pkl.exists() and _germ_pkl.exists():
    _f = pickle.load(open(_fid_pkl, "rb")); prep_fiducials, meas_fiducials = _f["prep"], _f["meas"]
    germs = pickle.load(open(_germ_pkl, "rb"))
    print("loaded cached fiducials/germs")
else:
    print("Selecting fiducials/germs -- this is the expensive, tune-able step (hurdle #1).")
    # Meas fiducials select cleanly; prep search may abort -- try longer candidates / greedy if so.
    prep_fiducials, meas_fiducials = _fs.find_fiducials(
        ideal, candidate_fid_counts={4: "all upto"}, algorithm="greedy", verbosity=1)
    germs = _gs.find_germs(ideal, candidate_germ_counts={3: "all upto", 4: "all upto"},
                           num_gs_copies=2, seed=0, verbosity=1)
    pickle.dump({"prep": prep_fiducials, "meas": meas_fiducials}, open(_fid_pkl, "wb"))
    pickle.dump(germs, open(_germ_pkl, "wb"))
print(f"{len(prep_fiducials)} prep / {len(meas_fiducials)} meas fiducials, {len(germs)} germs")

Selecting fiducials/germs -- this is the expensive, tune-able step (hurdle #1).
Initial Length Available Fiducial List: 11111
Length Available Fiducial List Dropped Identities and Duplicates: 2843
Using greedy algorithm.
Complete initial fiducial set succeeds.
Now searching for best fiducial set.
Starting fiducial list optimization. Lower score is better.
Acceptable candidate solution found.
Score: major=-64 minor=560.4098746094845, N: 64
Exiting greedy search.
Preparation fiducials:
['{}@(0,1,2)', 'Gypi2:0Gypi2:0Gcnot:2:1@(0,1,2)', 'Gxpi2:1Gxpi2:0Gxpi2:1@(0,1,2)', 'Gxpi2:0Gxpi2:2Gxpi2:2@(0,1,2)', 'Gxpi2:1Gxpi2:1Gcnot:1:2@(0,1,2)', 'Gxpi2:0Gxpi2:0Gcnot:0:1Gxpi2:0@(0,1,2)', 'Gxpi2:0Gxpi2:0Gcnot:0:1Gcnot:1:2@(0,1,2)', 'Gxpi2:1Gypi2:0Gxpi2:2@(0,1,2)', 'Gypi2:2Gypi2:0Gypi2:1@(0,1,2)', 'Gxpi2:1Gxpi2:0Gcnot:0:1Gypi2:2@(0,1,2)', 'Gxpi2:1Gxpi2:2Gypi2:0Gcnot:1:2@(0,1,2)', 'Gxpi2:1Gxpi2:2Gcnot:2:1@(0,1,2)', 'Gxpi2:0Gcnot:0:1Gypi2:0Gxpi2:2@(0,1,2)', 'Gypi2:2Gypi2:1Gcnot:1:0Gypi2:0@(0,1,2)', 'Gxpi

In [ ]:
# --- GST experiment design + circuit count ---
all_circuits = pygsti.circuits.create_lsgst_circuits(
    ideal, prep_fiducials, meas_fiducials, germs, MAX_LENGTHS)
print("full GST design:", len(all_circuits), "circuits  (x", OUTCOMES_PER_CIRCUIT, "outcomes)")

In [ ]:
# --- FPR: reduce the design (reuses the proven module) ---
fpr_reduction = fprmod.make_fpr_reduction_mask_function(
    base_model=fit_template, all_circuits=list(all_circuits), processor_spec=pspec,
    prep_fiducials=prep_fiducials, meas_fiducials=meas_fiducials, germs=germs,
    max_lengths=MAX_LENGTHS, outcomes_per_circuit=OUTCOMES_PER_CIRCUIT,
    stage1_method=FPR_STAGE1, stage2_method=FPR_STAGE2, verbose=True)
_mask = fpr_reduction(fit_template.to_vector())           # boolean mask over residual entries at the target
_kept = int(np.sum(_mask)); _tot = len(_mask)
print(f"FPR keeps {_kept}/{_tot} residuals ({100*_kept/_tot:.0f}%)  ->  ~{100*(1-_kept/_tot):.0f}% reduction")

In [ ]:
# --- compact weighted-LS oracle (residual + analytic Jacobian)  [VALIDATED on 1-qubit] ---
def prepare_dataset_arrays(circuits, dataset, template):
    """Fix a flat (circuit, outcome) ordering; precompute observed freqs f and weights sigma."""
    order, f_list, N_list = [], [], []
    for c in circuits:
        row = dataset[c]; total = row.total
        for o in template.sim.probs(c).keys():
            cnt = row[o] if o in row.outcomes else 0.0
            order.append((c, o)); f_list.append(cnt / total if total else 0.0); N_list.append(total)
    f = np.array(f_list); N = np.array(N_list)
    sigma = np.sqrt(np.clip(f * (1 - f) / np.maximum(N, 1), 1e-12, None))
    circ_outs = {}
    for i, (c, o) in enumerate(order):
        circ_outs.setdefault(c, []).append((o, i))
    return order, f, sigma, circ_outs

def residual_fn(x, template, circuits, f, sigma, circ_outs, mask=None):
    m = template.copy(); m.from_vector(np.asarray(x, float)); pr = m.sim.bulk_probs(circuits)
    F = np.empty(len(f))
    for c in circuits:
        for o, i in circ_outs[c]:
            F[i] = (pr[c][o] - f[i]) / sigma[i]
    return F[mask] if mask is not None else F

def jacobian_fn(x, template, circuits, sigma, circ_outs, mask=None):
    m = template.copy(); m.from_vector(np.asarray(x, float)); dp = m.sim.bulk_dprobs(circuits)
    J = np.empty((len(sigma), m.num_params))
    for c in circuits:
        for o, i in circ_outs[c]:
            J[i, :] = np.asarray(dp[c][o], float) / sigma[i]
    return J[mask] if mask is not None else J
print("oracle defined")

In [ ]:
# --- shot sweep: FPR vs non-FPR  [run on your compute] ---
# Solver: scipy least_squares (validated). To use POUNDERS instead, wrap the oracle as
#   fun(x) -> (residual_fn(...), jacobian_fn(...).T)   and call pounders.pouders(fun, x0, ...).
def run_one(circuits, dataset, mask=None, max_nfev=60):
    order, f, sigma, circ_outs = prepare_dataset_arrays(circuits, dataset, fit_template)
    x0 = fit_template.to_vector()
    res = least_squares(lambda x: residual_fn(x, fit_template, circuits, f, sigma, circ_outs, mask),
                        x0, jac=lambda x: jacobian_fn(x, fit_template, circuits, sigma, circ_outs, mask),
                        method="trf", max_nfev=max_nfev)
    dof = max((len(f) if mask is None else int(mask.sum())) - fit_template.num_params, 1)
    return res.x, float(res.fun @ res.fun), dof

rows = []
for shots in SHOT_SWEEP:
    ds = pygsti.data.simulate_data(truth, all_circuits, num_samples=int(shots),
                                   sample_error="multinomial", seed=DATA_SEED)
    for use_fpr in (False, True):
        mask = fpr_reduction(fit_template.to_vector()) if use_fpr else None
        active = list(all_circuits) if mask is None else [c for c in all_circuits]  # residual-level mask
        xhat, obj, dof = run_one(list(all_circuits), ds, mask=mask)
        n_active = len(mask) if mask is None else int(mask.sum())
        revealed = int(shots) * (len(all_circuits) if not use_fpr else n_active // OUTCOMES_PER_CIRCUIT)
        rows.append(dict(shots=int(shots), fpr=use_fpr, reduced_obj=obj / dof,
                         revealed_shots=revealed, x=xhat))
        print(f"shots={int(shots):>8} fpr={use_fpr!s:5}  reduced_obj={obj/dof:6.2f}  revealed~{revealed:.2e}")
sweep_df = pd.DataFrame(rows)

In [ ]:
# --- accuracy vs the true model + plots ---
# Gauge-optimize each estimate to `truth` and report per-gate diamond + infidelity (as in the 1Q study).
truth_full = truth.copy()
try: truth_full.set_all_parameterizations("full")
except Exception: pass
labels = list(truth.operations.keys())
def gname(l): return str(l)

def accuracy(xhat):
    est = fit_template.copy(); est.from_vector(xhat)
    try: est_full = est.copy(); est_full.set_all_parameterizations("full")
    except Exception: est_full = est
    go = pygsti.gaugeopt_to_target(est_full, truth_full)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        dia = np.nanmean([float(diamonddist(go.operations[l].to_dense(), truth_full.operations[l].to_dense(), truth_full.basis)) for l in labels])
        inf = np.mean([float(entanglement_infidelity(go.operations[l].to_dense(), truth_full.operations[l].to_dense(), truth_full.basis)) for l in labels])
    return dia, inf

acc = [accuracy(r.x) for _, r in sweep_df.iterrows()]
sweep_df["mean_diamond"], sweep_df["mean_infid"] = [a[0] for a in acc], [a[1] for a in acc]

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
for uf, sub in sweep_df.groupby("fpr"):
    sub = sub.sort_values("shots"); lab = "FPR" if uf else "full"
    ax[0].plot(sub["shots"], sub["mean_diamond"], "o-", label=lab)
    ax[1].plot(sub["shots"], sub["mean_infid"], "o-", label=lab)
    ax[2].plot(sub["shots"], sub["revealed_shots"], "o-", label=lab)
for a, t, yl in zip(ax, ["mean diamond to truth", "mean infidelity to truth", "revealed shots"],
                    ["diamond", "infidelity", "total shots"]):
    a.set_xscale("log"); a.set_yscale("log"); a.set_title(t); a.set_xlabel("shots per circuit"); a.set_ylabel(yl)
    a.grid(True, which="both", alpha=0.3); a.legend()
fig.suptitle("3-qubit GST scaffold: FPR vs full design"); plt.tight_layout(); plt.show()